# Part 1 - Distributed Data Processing with Spark

In [ ]:
from pathlib import Path
import requests
from pyspark.sql import SparkSession
import time
import pandas as pd

In [ ]:
# Define URL for required file
taxi_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"

# Create data/raw directory if it doesn't exist
BASE_DIR = Path.cwd().resolve()
data_dir = BASE_DIR / "data" / "raw"
data_dir.mkdir(parents=True, exist_ok=True)

# Defines File path for downloaded data
taxi_path = data_dir / "yellow_tripdata_2024-01.parquet"

# Download File and write to specified path
def download_file(url, path):
    if path.exists():
        return
     
    with requests.get(url, stream=True, timeout=30) as r:
        r.raise_for_status()
        with open(path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)

download_file(taxi_url, taxi_path)
print("\nFile downloaded successfully!")

In [ ]:
# Creating a Spark Session
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("LLM-Powered Applications & Distributed Computing") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

# Verifying the session creation
print(f'Spark version: {spark.version}')
print(f'App name: {spark.sparkContext.appName}')
print(f'Master: {spark.sparkContext.master}')
print(f'Default parallelism: {spark.sparkContext.defaultParallelism}')

In [ ]:
# Load the parquet file into a Spark Dataframe
df = spark.read.parquet(str(taxi_path))

# Display the schema: Column names and types
df.printSchema()

# Verify the first 5 rows of data
df.show(5, truncate=True)

# Display total number of rows
print(f"\nTotal number of rows: {df.count():,}")

# Display total number of columns
print(f"Total number of columns: {len(df.columns)}")

# Display total number of partitions
print(f"Total number of partitions: {df.rdd.getNumPartitions()}")

In [ ]:
# Timing for Reading Data with Spark
start = time.time()
df_spark = spark.read.parquet(str(taxi_path))
spark_read_time = time.time() - start

# Timing for Compution with Spark
start = time.time()
spark_count = df_spark.count()
spark_action_time = time.time() - start

# Timing for Reading Data with Pandas
start = time.time()
df_pandas = pd.read_parquet(str(taxi_path))
pandas_read_time = time.time() - start

# Display Results
print(f"Spark schema read (lazy): {spark_read_time:.3f}s")
print(f"Spark count action: {spark_action_time:.3f}s ({spark_count:,} rows)")
print(f"Pandas full read: {pandas_read_time:.3f}s ({len(df_pandas):,} rows)")
print(f"Pandas memory usage: {df_pandas.memory_usage(deep=True).sum() / 1e6:.1f} MB")

# Cleaning up Pandas Dataframe
del df_pandas